In [ ]:
import sys
import os
import json
import numpy as np
import scipy as sp
import pandas as pd
from matplotlib import pyplot as plt

from sklearn.decomposition import PCA


sys.path.append(os.path.join(os.getcwd(), '../../src/'))
from utils_worm import read_utils

sys.path.append(os.path.join(os.getcwd(), '../../../../'))
from utils import plot_utils
from utils import train_utils
from utils import analysis_utils
from models import model_utils

In [ ]:
import importlib
importlib.reload(read_utils)
importlib.reload(analysis_utils)
importlib.reload(plot_utils)
importlib.reload(train_utils)

In [ ]:
file_id = 19
organism = 'worm'
model_variant = 'ss'

selection_bhl = ['REV', 'VENTRAL_TURN', 'DORSAL_TURN', 'PAUSE']

neuron_labels = ['RMDDR', 'VB02', 'SMDDL', 'SMDDR', 'SIADR', 'DD01'] + ['AVBL', 'AVAR']

In [ ]:
filename, traces, eigenworms, discrete_states, (continuous_behaviors, kymograph) = analysis_utils.read_data(organism, file_id)
neuron_labels = [label for label in neuron_labels if label in traces.columns]

In [ ]:
bom_params, hgm_params = analysis_utils.read_models(read_utils.get_model_path(), filename, model_variant, manual=False)

((_, bom_man, bom_rec, bom_res, bom_pres, bom_resss, bom_bss),
 (_, hgm_man, hgm_rec, hgm_res, hgm_pres, hgm_resss, hgm_bss)) = analysis_utils.read_predictions(read_utils.get_model_path(), filename, model_variant, manual=False)

# manifold

In [ ]:
pca_modes, _, _, pca_vars_expl = train_utils.get_manifolds_pca([traces], scale=False, n_pc=hgm_man[0].values.shape[1])
cca_modes, _, _, _, _, _, cca_vars_expl, cca_vars_expl_rs = train_utils.get_manifolds_cca([traces], [discrete_states[selection_bhl]], scale=False, n_cv=hgm_man[0].values.shape[1], n_pc=discrete_states[selection_bhl].values.shape[1])

In [ ]:
bom_manifolds, bom_manifold_vars_expl = analysis_utils.postprocess_manifold(bom_man, traces, bom_params)
hgm_manifolds, hgm_manifold_vars_expl = analysis_utils.postprocess_manifold(bom_man, traces, bom_params)

In [ ]:
plot_utils.plot_manifolds([f'{str(file_id)}: {filename} s{s}' for s in range(len(bom_manifolds))],
						  [bom_manifolds[s] for s in range(len(bom_manifolds))],
						  [discrete_states for s in range(len(bom_manifolds))],
						  bom_manifold_vars_expl, None,
						  title='bom manifold', plots_per_row=len(bom_manifolds), top_space=3.5,
						  save_path=None)

In [ ]:
plot_utils.plot_manifolds([f'{str(file_id)}: {filename} s{s}' for s in range(len(hgm_manifolds))],
						  [hgm_manifolds[s] for s in range(len(hgm_manifolds))],
						  [discrete_states for s in range(len(hgm_manifolds))],
						  hgm_manifold_vars_expl, None,
						  title='hgm manifold', plots_per_row=len(hgm_manifolds), top_space=3.5,
						  save_path=None)

In [ ]:
plot_utils.plot_manifolds([f'{str(file_id)}: {filename} s{s}' for s in range(len(hgm_manifolds))],
						  [pca_modes[s] for s in range(len(hgm_manifolds))],
						  [discrete_states for s in range(len(hgm_manifolds))],
						  pca_vars_expl, None,
						  title='pca manifold', plots_per_row=len(hgm_manifolds), top_space=3.5,
						  save_path=None)

In [ ]:
plot_utils.plot_manifolds([f'{str(file_id)}: {filename} s{s}' for s in range(len(hgm_manifolds))],
						  [cca_modes[s] for s in range(len(hgm_manifolds))],
						  [discrete_states for s in range(len(hgm_manifolds))],
						  cca_vars_expl, cca_vars_expl_rs,
						  title='cca manifold', plots_per_row=len(hgm_manifolds), top_space=3.5,
						  save_path=None)

# RES MODES

In [ ]:
pca_pres = PCA(n_components=hgm_pres[0].values.shape[1])
pca_pres.fit(hgm_pres[0].values)

In [ ]:
modes = 5

plt.plot([sum(pca_pres.explained_variance_ratio_[:k]) for k in range(modes+1)])
sum(pca_pres.explained_variance_ratio_[:modes+1])

In [ ]:
cols = [f'Mode {i+1}' for i in range(hgm_pres[0].shape[1])]
cols_plot = cols[:modes]

bom_pres_modes = [pd.DataFrame(pca_pres.transform(bom_pres[0].values), columns=cols)]
hgm_pres_modes = [pd.DataFrame(pca_pres.transform(hgm_pres[0].values), columns=cols)]

bom_res_modes = [pd.DataFrame(pca_pres.transform(bom_res[0].values), columns=cols)]
hgm_res_modes = [pd.DataFrame(pca_pres.transform(hgm_res[0].values), columns=cols)]

# MANIFOLD COLORING

In [ ]:
sigmoid = lambda x: 1 / (1 + np.exp(-x))

In [ ]:
hgm_man_zs = (hgm_man[0].values - hgm_man[0].values.mean(axis=0)) / np.where((N_std := hgm_man[0].values.std(axis=0)) == 0.0, 1.0, N_std)
WM = hgm_params[0]['model']['WM']
VB = hgm_params[0]['model']['VB']

gating_space = sigmoid(hgm_man_zs @ WM) @ VB
gating_space_modes = [pd.DataFrame(pca_pres.transform(gating_space), columns=cols)]

In [ ]:
resss_ids = [hgm_resss[0].columns.get_loc(ss) for ss in hgm_resss[0].columns]

plot_utils.plot_manifolds_gradient([f'Mode {ss+1}' for ss in range(modes)],
								   [hgm_manifolds[0]] * modes,
								   [gating_space_modes[0][[f'Mode {ss+1}']] for ss in range(modes)],
								   hgm_manifold_vars_expl * modes, None,
								   title=f'{str(file_id)}: {filename} gcamp manifolds by gating of the different modes',
								   plots_per_row=3, top_space=1,
								   save_path=None)

In [ ]:
plot_utils.plot_manifolds_gradient([f'Mode {ss+1}' for ss in range(modes)],
								   [pca_modes[0]] * modes,
								   [gating_space_modes[0][[f'Mode {ss+1}']] for ss in range(modes)],
								   pca_vars_expl * modes, None,
								   title=f'{str(file_id)}: {filename} gcamp manifolds by gating of the different modes',
								   plots_per_row=3, top_space=1,
								   save_path=None)

In [ ]:
plot_utils.plot_manifolds_gradient([f'Mode {ss+1}' for ss in range(modes)],
								   [cca_modes[0]] * modes,
								   [gating_space_modes[0][[f'Mode {ss+1}']] for ss in range(modes)],
								   cca_vars_expl * modes, cca_vars_expl_rs * modes,
								   title=f'{str(file_id)}: {filename} gcamp manifolds by gating of the different modes',
								   plots_per_row=3, top_space=1,
								   save_path=None)

In [ ]:
# max_x = 500
max_x = bom_pres[0].values.shape[0]

plot_utils.plot_traces_splits_pairs(
	[hgm_res_modes[0].loc[:max_x, cols_plot], hgm_pres_modes[0].loc[:max_x, cols_plot]],
	# [bom_res_modes[0].loc[:max_x, cols_plot], bom_pres_modes[0].loc[:max_x, cols_plot]],
	None,
	gating_space_modes[0].loc[:max_x, cols_plot],
	discrete_states.loc[:max_x, ['REV']], label='reversal',
	title=f'{filename} gcamp hgm vs bom residual PC modes with gating', ylimits=None, plots_per_row=1,
	save_path=None)  #'../../out/plots/GCAMP_RES'

In [ ]:
# # max_x = 500
# max_x = bom_pres[0].values.shape[0]
#
# plot_utils.plot_traces_splits(
# 	[hgm_res_modes[0].loc[:max_x, cols_plot]],
# 	[bom_res_modes[0].loc[:max_x, cols_plot]],
# 	gating_space_modes[0].loc[:max_x, cols_plot],
# 	discrete_states.loc[:max_x, ['REV']], label='reversal',
# 	title=f'{filename} gcamp hgm vs bom residual PC modes with gating', ylimits=None, plots_per_row=1,
# 	save_path=None)  #'../../out/plots/GCAMP_RES'

In [ ]:
# # max_x = 500
# max_x = bom_pres[0].values.shape[0]
#
# plot_utils.plot_traces_splits(
# 	[hgm_pres_modes[0].loc[:max_x, cols_plot]],
# 	[bom_pres_modes[0].loc[:max_x, cols_plot]],
# 	gating_space_modes[0].loc[:max_x, cols_plot],
# 	discrete_states.loc[:max_x, ['REV']], label='reversal',
# 	title=f'{filename} gcamp hgm gating space PC modes', ylimits=None, plots_per_row=1,
# 	save_path=None)  #'../../out/plots/GCAMP_RES'

# traces

In [ ]:
# max_x = 500
max_x = traces.values.shape[0]

plot_utils.plot_traces_splits(
	[hgm_rec[i].loc[:max_x, neuron_labels] for i in range(len(hgm_rec))],
	[bom_rec[i].loc[:max_x, neuron_labels] for i in range(len(hgm_rec))],
	traces.loc[:max_x, neuron_labels],
	discrete_states.loc[:max_x, ['REV']], label='reversal',
	title=f'{filename} gcamp hgm & bom definitions manifold', ylimits=None, plots_per_row=1,
	save_path=None)  #'../../out/plots/GCAMP_MAN'

In [ ]:
# max_x = 500
max_x = traces.values.shape[0]

plot_utils.plot_traces_splits(
	[hgm_res[0].loc[:max_x, neuron_labels]],
	[bom_res[0].loc[:max_x, neuron_labels]],
	traces.loc[:max_x, neuron_labels],
	discrete_states.loc[:max_x, ['REV']], label='reversal',
	title=f'{filename} gcamp hgm & bom definitions residuals', ylimits=None, plots_per_row=1,
	save_path=None)  #'../../out/plots/GCAMP_RES'

In [ ]:
# max_x = 500
max_x = traces.values.shape[0]


plot_utils.plot_traces_splits_pairs(
	[hgm_res[0].loc[:max_x, neuron_labels],
	 hgm_pres[0].loc[:max_x, neuron_labels]],
	[bom_res[0].loc[:max_x, neuron_labels],
	 bom_pres[0].loc[:max_x, neuron_labels]],
	traces.loc[:max_x, neuron_labels],
	discrete_states.loc[:max_x, ['REV']], label='reversal',
	title=f'{filename} gcamp hgm & bom predictions residuals', ylimits=None, plots_per_row=1,
	save_path=None)  #../../out/plots/f'GCAMP_PRES_S{i}'

In [ ]:
# max_x = 500
max_x = traces.values.shape[0]

plot_utils.plot_traces_splits(
	[hgm_bss[0].loc[:max_x]],
	[bom_bss[0].loc[:max_x]],
	None,
	discrete_states.loc[:max_x, ['REV']], label='reversal',
	title=f'{filename} gcamp hgm & bom behavioral subspace', ylimits=None, plots_per_row=1,
	save_path=None)  #'../../out/plots/GCAMP_RES'

In [ ]:
# max_x = 500
max_x = traces.values.shape[0]

plot_utils.plot_traces_splits(
	[hgm_resss[0].loc[:max_x]],
	[bom_resss[0].loc[:max_x]],
	None,
	discrete_states.loc[:max_x, ['REV']], label='reversal',
	title=f'{filename} gcamp hgm & bom residual subspace', ylimits=None, plots_per_row=1,
	save_path=None)  #'../../out/plots/GCAMP_RES'